# Motivation: Why Photometric Error Calibration Matters

Type Ia supernovae (SNe Ia) are standard candles used to measure cosmological distances.
Their standardized brightness is derived from the SALT light-curve fit parameters:
* **x0** — amplitude (encodes distance modulus: μ ∝ −2.5 log x0)
* **x1** — stretch (brighter-slower relation)
* **c** — color (brighter-bluer relation)

The Tripp estimator gives the distance modulus as:
$$\mu = -2.5\log_{10}(x_0) + \alpha\,x_1 - \beta\,c + \text{const}$$

A key question: **what happens if the reported photometric errors are wrong?**
This notebook simulates a single SN Ia light curve with [LightCurveLynx](https://lightcurvelynx.readthedocs.io)
and LSST passbands, fits it with the SALT3 model via [sncosmo](https://sncosmo.readthedocs.io),
then repeats the fit after inflating the errors by a fixed floor of 0.05 mag.
The comparison directly motivates the need for `uncle-val`'s photometric uncertainty calibration.

## Setup

In [ ]:
%pip install astropy iminuit lightcurvelynx matplotlib numpy pandas sncosmo

In [ ]:
import warnings

import astropy.table
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sncosmo
from lightcurvelynx.astro_utils.passbands import PassbandGroup
from lightcurvelynx.models.sncosmo_models import SncosmoWrapperModel
from lightcurvelynx.obstable.lsst_obstable import LSSTObsTable
from lightcurvelynx.simulate import simulate_lightcurves

### Load LSST passbands and register them with sncosmo

LightCurveLynx downloads and caches the official LSST throughput curves from
the [lsst/throughputs](https://github.com/lsst/throughputs) repository.

* **Passband loading** (`PassbandGroup`) is used by LightCurveLynx for *simulation*.
* **sncosmo registration** is needed only for *fitting* — it ensures `sncosmo.fit_lc`
  uses the exact same throughput curves that were used to generate the data.

In [ ]:
BANDS = ["u", "g", "r", "i", "z", "y"]
BAND_COLORS = {"u": "#377eb8", "g": "#4daf4a", "r": "#e41a1c", "i": "#984ea3", "z": "#ff7f00", "y": "#a65628"}

# Zero-point for nJy fluxes on the AB system: zp = 2.5 * log10(3.631e12) ≈ 31.4
ZP = 8.9 + 2.5 * 9  # = 31.4

# Zero-point calibration uncertainty to simulate (magnitudes)
ZP_ERR_MAG = 0.05

# Load all LSST passbands — used by LightCurveLynx for simulation
passbands = PassbandGroup.from_preset("lsst", filters=BANDS)

# Register with sncosmo — required only for fitting (sncosmo.fit_lc)
for f, passband in passbands.passbands.items():
    bp = sncosmo.Bandpass(*passband.normalized_system_response.T, name=f"lightcurvelynx_{f}")
    sncosmo.register(bp, force=True)

print("Registered sncosmo bands:", [f"lightcurvelynx_{f}" for f in passbands.passbands])

## Step 1: Simulate a SN Ia light curve

### True SN Ia parameters

We place a fast-declining (x1 = −1) SN Ia at z = 0.1 with a typical peak absolute
magnitude M_B ≈ −19.3. The amplitude x0 is derived from M_B and the luminosity distance
at z = 0.1 (flat ΛCDM, H0 = 70, Ωm = 0.3 → μ ≈ 38.3 mag).

In [ ]:
TRUE_PARAMS = dict(
    z=0.1,
    t0=60630.0,  # MJD peak — early in the DP1 window to maximise post-peak coverage
    x0=4.32e-4,  # amplitude: derived from M_B ≈ -19.3 + distance modulus at z=0.1
    x1=-1.0,  # stretch (fast-declining / "short" SN)
    c=0.05,  # color
)
# Place the SN in the dense DP1 field (DEEP_A1 / COSMOS area)
SN_RA, SN_DEC = 53.0, -28.0

print("True parameters:", TRUE_PARAMS)
print(f"SN position: RA={SN_RA}, Dec={SN_DEC}")

### Load the Rubin DP1 CCD visit table

We use real DP1 observation metadata (pointing, seeing, sky background, zero-points)
via LightCurveLynx's `LSSTObsTable`.
We set `zp_err_mag=0.05` to simulate a 5% zero-point calibration uncertainty —
a realistic miscalibration scenario for a new survey.

In [ ]:
ccdvisit = pd.read_parquet("https://data.lsdb.io/hats/dp1/ccd_visit.parquet")
obstable = LSSTObsTable.from_ccdvisit_table(ccdvisit, zp_err_mag=ZP_ERR_MAG)
print(
    f"Loaded {len(ccdvisit)} CCD visits, MJD {ccdvisit['expMidptMJD'].min():.1f} – {ccdvisit['expMidptMJD'].max():.1f}"
)

In [ ]:
# SncosmoWrapperModel wraps the SALT3 sncosmo source in the LightCurveLynx framework.
source = SncosmoWrapperModel(
    "salt3",
    t0=TRUE_PARAMS["t0"],
    x0=TRUE_PARAMS["x0"],
    x1=TRUE_PARAMS["x1"],
    c=TRUE_PARAMS["c"],
    redshift=TRUE_PARAMS["z"],
    ra=SN_RA,
    dec=SN_DEC,
    node_label="source",
)
print("SncosmoWrapperModel source:", source.source_name)

### Simulate using LightCurveLynx

`simulate_lightcurves` handles everything in one call: parameter sampling,
bandflux evaluation via `SncosmoWrapperModel`, noise from the `FakeObsTable`,
and returns a `NestedFrame` with one row per object and a nested `lightcurve` table.

In [ ]:
results = simulate_lightcurves(
    source,
    num_samples=1,
    obstable=obstable,
    passbands=passbands,
    obs_time_window_offset=(-10, 30),  # observer-frame days around t0 (DP1 spans ~-7 to +26)
    rng=np.random.default_rng(42),
    progress_bar=False,
)

# Extract the light curve — nested "lightcurve" table: mjd, filter, flux, fluxerr, flux_perfect
lc = results.iloc[0]["lightcurve"]

# Drop the rare rows where the noise model could not produce a valid error
lc = lc.dropna(subset=["fluxerr"]).reset_index(drop=True)

times = lc["mjd"].values
obs_bands = lc["filter"].values
flux_noisy = lc["flux"].values
flux_err = lc["fluxerr"].values
flux_perfect = lc["flux_perfect"].values

print(
    f"Simulated {len(lc)} observations across bands: { {b: int((obs_bands==b).sum()) for b in BANDS if (obs_bands==b).any()} }"
)
print(f"Peak flux (r-band) ≈ {flux_perfect[obs_bands == 'r'].max():.0f} nJy")

In [ ]:
# ZP error contribution: σ_ZP = |F_true| * Δm_ZP * ln(10)/2.5
# Uses perfect flux because ZP error is applied to the true source flux in the noise model
zp_flux_err = np.abs(flux_perfect) * ZP_ERR_MAG * np.log(10) / 2.5

# Reported (photon-noise only) errors — what a pipeline ignoring ZP calibration would give
flux_err_reported = np.sqrt(np.maximum(flux_err**2 - zp_flux_err**2, 0.0))

### Plot the simulated light curve

In [ ]:
fig, (ax_top, ax_bot) = plt.subplots(
    2,
    1,
    figsize=(8, 5.5),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05},
)

# Dense time grid for smooth noiseless model lines
t_fine = np.linspace(times.min() - 2, times.max() + 2, 400)
snc_true = sncosmo.Model("salt3")
snc_true.set(**TRUE_PARAMS)

active_bands = [b for b in BANDS if (obs_bands == b).any()]
for band in active_bands:
    mask = obs_bands == band
    ax_top.errorbar(
        times[mask],
        flux_noisy[mask],
        yerr=flux_err[mask],
        fmt="o",
        ms=3,
        color=BAND_COLORS[band],
        label=f"LSST-{band}",
        capsize=2,
        elinewidth=0.8,
    )
    flux_fine = snc_true.bandflux(f"lightcurvelynx_LSST_{band}", t_fine, zp=ZP, zpsys="ab")
    ax_top.plot(t_fine, flux_fine, "-", color=BAND_COLORS[band], alpha=0.5, lw=1.5)

    # Lower panel: ratio true / reported errors per band
    ratio = flux_err[mask] / flux_err_reported[mask]
    ax_bot.scatter(times[mask], ratio, s=4, color=BAND_COLORS[band])

ax_top.set_ylim(-2000, flux_perfect.max() * 1.25)
ax_top.set_ylabel("Flux (nJy)")
ax_top.set_title(
    f"Simulated SN Ia (DP1 noise) — SALT3, z={TRUE_PARAMS['z']},"
    f" x1={TRUE_PARAMS['x1']}, c={TRUE_PARAMS['c']}"
)
ax_top.legend(ncol=3)

ax_bot.axhline(1, color="k", lw=0.8, ls="--")
ax_bot.set_ylabel(r"$\sigma_{\rm true} / \sigma_{\rm reported}$")
ax_bot.set_xlabel("MJD")
ax_bot.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

In [ ]:
# Map simulation filter names to registered sncosmo band names
sncosmo_bands = np.array([f"lightcurvelynx_LSST_{b}" for b in obs_bands])


def make_sncosmo_table(flux, flux_err):
    """Build an astropy Table suitable for sncosmo.fit_lc."""
    return astropy.table.Table(
        {
            "time": times,
            "band": sncosmo_bands,
            "flux": flux,
            "fluxerr": flux_err,
            "zp": np.full(len(times), ZP),
            "zpsys": ["ab"] * len(times),
        }
    )


lc_table = make_sncosmo_table(flux_noisy, flux_err)

In [ ]:
model = sncosmo.Model("salt3")
model.set(z=TRUE_PARAMS["z"])  # fix redshift (known from spectroscopy)

# Initial guess close to truth to speed up convergence
model.set(t0=TRUE_PARAMS["t0"], x0=TRUE_PARAMS["x0"], x1=0.0, c=0.0)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    result1, fitted_model1 = sncosmo.fit_lc(
        lc_table,
        model,
        ["t0", "x0", "x1", "c"],
        bounds={"x1": (-3, 3), "c": (-0.5, 0.5)},
    )

print("Fit 1 (true errors, photon + ZP):")
print(f"  chi2/dof = {result1.chisq:.2f} / {result1.ndof}  (reduced = {result1.chisq/result1.ndof:.2f})")
for p, v in zip(result1.param_names, result1.parameters):
    e = result1.errors.get(p)  # None for fixed params (z)
    truth = TRUE_PARAMS.get(p)
    e_str = f"± {e:.4g}" if e is not None else "(fixed)"
    truth_str = f"  [truth: {truth:.4g}]" if truth is not None else ""
    print(f"  {p:3s} = {v:+.4g}  {e_str}{truth_str}")

## Step 3: Fit with reported errors (ZP uncertainty excluded)

A pipeline that does not account for zero-point calibration uncertainty will report
only the photon-noise component. We recover these "reported" errors by removing
the ZP contribution in quadrature:

$$\sigma_{\rm reported} = \sqrt{\sigma_{\rm true}^2 - \sigma_{\rm ZP}^2}$$

where $\sigma_{\rm ZP} = \frac{\ln 10}{2.5}\,|F|\,\Delta m_{\rm ZP}$
with $\Delta m_{\rm ZP} = 0.05$ mag.
Fitting with under-estimated errors biases the SALT3 parameters.

In [ ]:
# flux_err_reported was computed above (photon-noise only, ZP contribution removed)
print(f"Median correction factor: {np.median(flux_err_reported / flux_err):.2f}")
print(f"Min correction factor: {np.min(flux_err_reported / flux_err):.4f}")

In [ ]:
lc_table_reported = make_sncosmo_table(flux_noisy, flux_err_reported)

model2 = sncosmo.Model("salt3")
model2.set(z=TRUE_PARAMS["z"], t0=TRUE_PARAMS["t0"], x0=TRUE_PARAMS["x0"], x1=0.0, c=0.0)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    result2, fitted_model2 = sncosmo.fit_lc(
        lc_table_reported,
        model2,
        ["t0", "x0", "x1", "c"],
        bounds={"x1": (-3, 3), "c": (-0.5, 0.5)},
    )

print("Fit 2 (reported errors, photon noise only):")
print(f"  chi2/dof = {result2.chisq:.2f} / {result2.ndof}  (reduced = {result2.chisq/result2.ndof:.2f})")
for p, v in zip(result2.param_names, result2.parameters):
    e = result2.errors.get(p)
    truth = TRUE_PARAMS.get(p)
    e_str = f"± {e:.4g}" if e is not None else "(fixed)"
    truth_str = f"  [truth: {truth:.4g}]" if truth is not None else ""
    print(f"  {p:3s} = {v:+.4g}  {e_str}{truth_str}")

## Step 4: Compare the two fits

In [ ]:
# Replace x0 with -2.5*log10(x0), the quantity that enters the distance modulus,
# so the y-axis is in human-readable magnitudes.


def x0_to_mag(x0):
    return -2.5 * np.log10(x0)


# Parameters to display and their derived uncertainties (via error propagation for x0→mag)
def get_val_err(result, pname):
    idx = list(result.param_names).index(pname)
    v = result.parameters[idx]
    e = result.errors.get(pname, 0.0)
    if pname == "x0":
        e_mag = 2.5 / np.log(10) * e / v  # σ_{-2.5 log x0} = 2.5/ln10 * σ_x0/x0
        return x0_to_mag(v), e_mag
    return v, e


PLOT_PARAMS = {
    "t0": ("t0 (days)", TRUE_PARAMS["t0"]),
    "x0": ("−2.5 log₁₀(x0) (mag)", x0_to_mag(TRUE_PARAMS["x0"])),
    "x1": ("x1", TRUE_PARAMS["x1"]),
    "c": ("c", TRUE_PARAMS["c"]),
}

fig, axes = plt.subplots(2, 2, figsize=(8, 6))
axes_flat = axes.flatten()

for ax, (pname, (label, truth)) in zip(axes_flat, PLOT_PARAMS.items()):
    v1, e1 = get_val_err(result1, pname)
    v2, e2 = get_val_err(result2, pname)

    ax.errorbar([0], [v1], yerr=[e1], fmt="o", color="steelblue", ms=8, capsize=5, label="True errors")
    ax.errorbar([1], [v2], yerr=[e2], fmt="s", color="tomato", ms=8, capsize=5, label="Reported errors")
    ax.axhline(truth, color="k", lw=1.5, ls="--", label="Truth")

    ax.set_title(label, fontsize=10)
    ax.set_xticks([])
    ax.set_xlim(-0.5, 1.5)

axes_flat[0].legend(fontsize=8)
fig.suptitle("SALT3 parameter recovery: true vs. reported errors", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Compute distance modulus offset (Tripp relation)
ALPHA, BETA = 0.14, 3.1


def tripp_mu_offset(params, errors):
    """Δμ = -2.5/ln(10) * Δx0/x0 (shift relative to truth)."""
    idx_x0 = list(params.keys()).index("x0")
    idx_x1 = list(params.keys()).index("x1")
    idx_c = list(params.keys()).index("c")
    vals = list(params.values())
    x0_fit = vals[idx_x0]
    x1_fit = vals[idx_x1]
    c_fit = vals[idx_c]
    mu_fit = -2.5 * np.log10(x0_fit) + ALPHA * x1_fit - BETA * c_fit
    x0_true = TRUE_PARAMS["x0"]
    x1_true = TRUE_PARAMS["x1"]
    c_true = TRUE_PARAMS["c"]
    mu_true = -2.5 * np.log10(x0_true) + ALPHA * x1_true - BETA * c_true
    return mu_fit - mu_true


params1 = dict(zip(result1.param_names, result1.parameters))
params2 = dict(zip(result2.param_names, result2.parameters))

dmu1 = tripp_mu_offset(params1, result1.errors)
dmu2 = tripp_mu_offset(params2, result2.errors)

print("Distance modulus offset (Δμ = μ_fit − μ_true):")
print(f"  True errors     :  Δμ = {dmu1:+.4f} mag")
print(f"  Reported errors :  Δμ = {dmu2:+.4f} mag")
print()
print(f"Goodness of fit (χ²/dof):")
print(f"  True errors     :  {result1.chisq:.1f} / {result1.ndof} = {result1.chisq/result1.ndof:.2f}")
print(f"  Reported errors :  {result2.chisq:.1f} / {result2.ndof} = {result2.chisq/result2.ndof:.2f}")

## Discussion

The two fits use **identical photometric measurements** but different error estimates.

* **Fit 1 (true errors)** uses the errors produced by LightCurveLynx with ,
  which includes a 5% zero-point calibration uncertainty added in quadrature to the photon noise.
  These represent what the errors *should* be.
* **Fit 2 (reported errors)** uses only the photon-noise component — what a pipeline
  that ignores ZP calibration uncertainty would actually report.

Key observations:
- **χ²/dof > 1** for the reported errors — the fitter is over-confident because
  the errors are under-estimated, forcing the fit to chase noise.
- **Δμ** shifts between the two fits, illustrating a direct cosmological bias
  from under-estimated photometric uncertainties.

In real surveys the pipeline's reported errors systematically miss contributions
such as ZP calibration, flat-fielding, and atmospheric chromatic effects —
causing χ²/dof > 1 and biased Tripp parameter recovery.

**This is precisely what  addresses**: it learns a per-observation correction
factor *u* such that , ensuring that the whitened
residuals of constant stars follow *N*(0, 1) — guaranteeing unbiased parameter recovery.